# Menyiapkan Knowledge Base RAG di DuckDB

Notebook ini menyiapkan *knowledge base* untuk sistem RAG: teks mentah dipotong (chunking), diubah jadi vektor (embedding), disimpan di DuckDB dengan index HNSW, lalu diuji lewat semantic search dan RAG sederhana memakai Qwen2.5-3B lokal.

**Isi notebook:**
1. Setup DuckDB + ekstensi `vss`
2. Skema tabel dengan ID deterministik (ingestion idempotent)
3. Embedding multilingual (384 dimensi, dinormalisasi)
4. Chunking dengan validasi parameter
5. Ingest teks dan PDF, termasuk uji "jalankan dua kali"
6. Semantic search dengan cosine similarity
7. RAG dengan chat template, stop yang benar, dan similarity threshold
8. Simpan `knowledge.duckdb` ke Google Drive

## 0. Setup Environment

`llama-cpp-python` di-build dari source saat install (versi CPU), jadi sel ini bisa memakan waktu beberapa menit.

In [1]:
!pip -q install duckdb sentence-transformers pypdf llama-cpp-python

## 1. Konsep: Kenapa Vector Database?
- **Keyword search** cocok untuk pencocokan kata persis.
- **Semantic search** (vektor) menangkap *makna*, bukan hanya kata. Ini inti tahap retrieval di RAG.
- DuckDB dipilih karena **embedded** (satu file `.duckdb`, tanpa server terpisah) dan punya ekstensi `vss` untuk index HNSW.

> Catatan: menyimpan index HNSW ke file masih berstatus **eksperimental** di DuckDB (`hnsw_enable_experimental_persistence`). Untuk lab ini aman, tapi kalau proses berhenti tidak bersih, index bisa perlu dibuat ulang.

In [2]:
import duckdb

DB_PATH = "knowledge.duckdb"
con = duckdb.connect(DB_PATH)
con.execute("INSTALL vss; LOAD vss;")
con.execute("SET hnsw_enable_experimental_persistence=true;")
print(con.execute("PRAGMA database_list").fetchall())

[(570, 'knowledge', '/content/knowledge.duckdb')]


## 2. Skema Tabel `documents`

Dua keputusan desain penting:
- **`id` = hash SHA-256 dari isi chunk.** Chunk yang sama selalu menghasilkan ID yang sama, sehingga menjalankan ulang ingest tidak menggandakan data.
- **Index HNSW memakai metrik `cosine`.** Metrik index harus sama dengan fungsi jarak di query supaya index benar-benar dipakai.

In [3]:
EMBED_DIM = 384

con.execute(f"""
CREATE TABLE IF NOT EXISTS documents (
    id VARCHAR PRIMARY KEY,
    source VARCHAR,
    content TEXT,
    embedding FLOAT[{EMBED_DIM}],
    created_at TIMESTAMP DEFAULT current_timestamp
);
""")
con.execute(
    "CREATE INDEX IF NOT EXISTS idx_emb ON documents USING HNSW (embedding) WITH (metric = 'cosine');"
)
print("Tabel documents siap.")

Tabel documents siap.


## 3. Load Embedding Model

Model yang dipakai: `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensi). Model populer `all-MiniLM-L6-v2` juga 384 dimensi, tapi hanya dilatih untuk bahasa Inggris, jadi kurang cocok untuk teks Indonesia.

Embedding dinormalisasi (`normalize_embeddings=True`) supaya panjang vektornya 1. Dengan begitu skor cosine mudah dibaca: mendekati 1 berarti sangat mirip.

In [4]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
assert embed_model.get_sentence_embedding_dimension() == EMBED_DIM

def embed(texts):
    return embed_model.encode(texts, normalize_embeddings=True)

print(embed("contoh kalimat").shape)

(384,)


In [5]:
sentences = [
    "Saya sedang belajar natural language processing",
    "Belajar AI menyenangkan sekali untuk meningkatkan pemahaman teknologi",
]
embeddings = embed(sentences)
print(embeddings.shape)
print(embed_model.similarity(embeddings, embeddings))

(2, 384)
tensor([[1.0000, 0.3541],
        [0.3541, 1.0000]])


## 4. Contoh Data Mentah (Bahasa Indonesia)

In [6]:
raw_documents = [
    {"source": "faq_produk.txt", "text": "Garansi produk kami berlaku selama 1 tahun sejak "
        "tanggal pembelian. Klaim garansi dapat dilakukan melalui aplikasi atau toko resmi."},
    {"source": "sop_retur.txt", "text": "Proses retur barang maksimal 7 hari setelah barang "
        "diterima. Barang harus dalam kondisi asli dan menyertakan bukti pembelian."},
    {"source": "faq_pengiriman.txt", "text": "Pengiriman reguler memakan waktu 2-4 hari kerja. "
        "Pengiriman ekspres tersedia untuk kota-kota besar dengan estimasi 1 hari kerja."},
]
display(raw_documents)

[{'source': 'faq_produk.txt',
  'text': 'Garansi produk kami berlaku selama 1 tahun sejak tanggal pembelian. Klaim garansi dapat dilakukan melalui aplikasi atau toko resmi.'},
 {'source': 'sop_retur.txt',
  'text': 'Proses retur barang maksimal 7 hari setelah barang diterima. Barang harus dalam kondisi asli dan menyertakan bukti pembelian.'},
 {'source': 'faq_pengiriman.txt',
  'text': 'Pengiriman reguler memakan waktu 2-4 hari kerja. Pengiriman ekspres tersedia untuk kota-kota besar dengan estimasi 1 hari kerja.'}]

## 5. Chunking

Strategi *fixed-size chunking* berbasis kata dengan overlap. Fungsi ini sekarang menolak konfigurasi `overlap >= size`: tanpa validasi, langkah geser jatuh ke 1 kata dan hasilnya belasan chunk yang hampir identik tanpa ada error sama sekali.

In [7]:
def chunk_text(text, size=60, overlap=15):
    if size <= 0 or not (0 <= overlap < size):
        raise ValueError(f"Butuh size > 0 dan 0 <= overlap < size (dapat size={size}, overlap={overlap})")
    words = text.split()
    step = size - overlap
    chunks = []
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i:i + size]))
        if i + size >= len(words):
            break
    return chunks

chunk_text(raw_documents[0]["text"])

['Garansi produk kami berlaku selama 1 tahun sejak tanggal pembelian. Klaim garansi dapat dilakukan melalui aplikasi atau toko resmi.']

### Eksperimen ukuran chunk pada `raw_documents[1]`

In [8]:
teks = raw_documents[1]["text"]
print(f"Jumlah kata: {len(teks.split())}\n")

for size, overlap in [(60, 15), (10, 3), (6, 2)]:
    hasil = chunk_text(teks, size=size, overlap=overlap)
    print(f"size={size}, overlap={overlap} -> {len(hasil)} chunk")
    for c in hasil:
        print("   -", c)
    print()

# Konfigurasi tidak valid sekarang langsung ketahuan
for size, overlap in [(5, 15), (6, 8)]:
    try:
        chunk_text(teks, size=size, overlap=overlap)
    except ValueError as e:
        print("Ditolak:", e)

Jumlah kata: 18

size=60, overlap=15 -> 1 chunk
   - Proses retur barang maksimal 7 hari setelah barang diterima. Barang harus dalam kondisi asli dan menyertakan bukti pembelian.

size=10, overlap=3 -> 3 chunk
   - Proses retur barang maksimal 7 hari setelah barang diterima. Barang
   - barang diterima. Barang harus dalam kondisi asli dan menyertakan bukti
   - dan menyertakan bukti pembelian.

size=6, overlap=2 -> 4 chunk
   - Proses retur barang maksimal 7 hari
   - 7 hari setelah barang diterima. Barang
   - diterima. Barang harus dalam kondisi asli
   - kondisi asli dan menyertakan bukti pembelian.

Ditolak: Butuh size > 0 dan 0 <= overlap < size (dapat size=5, overlap=15)
Ditolak: Butuh size > 0 dan 0 <= overlap < size (dapat size=6, overlap=8)


## 6. Ingest: Embed & Simpan ke DuckDB

Ingest dibuat **idempotent**: chunk yang ID-nya sudah ada dilewati. Nama file juga dinormalisasi, karena Colab menamai ulang file yang diunggah dua kali menjadi `nama (1).pdf`.

In [9]:
import hashlib, re

def normalize_source(name):
    return re.sub(r"\s\(\d+\)(?=\.[^.]+$)", "", name)

def chunk_id(chunk):
    return hashlib.sha256(" ".join(chunk.split()).encode("utf-8")).hexdigest()

def ingest(text, source, size=60, overlap=15):
    source = normalize_source(source)
    chunks = chunk_text(text, size=size, overlap=overlap)
    baru = [c for c in chunks
            if con.execute("SELECT 1 FROM documents WHERE id = ?", [chunk_id(c)]).fetchone() is None]
    if baru:
        vektor = embed(baru)
        for c, v in zip(baru, vektor):
            con.execute(
                "INSERT INTO documents VALUES (?, ?, ?, ?, current_timestamp) ON CONFLICT DO NOTHING",
                [chunk_id(c), source, c, v.tolist()],
            )
    return len(baru), len(chunks) - len(baru)

def jumlah_baris():
    return con.execute("SELECT COUNT(*) FROM documents").fetchone()[0]

for doc in raw_documents:
    masuk, dilewati = ingest(doc["text"], doc["source"])
    print(f"{doc['source']}: {masuk} chunk baru, {dilewati} dilewati")
print("Total baris:", jumlah_baris())

faq_produk.txt: 1 chunk baru, 0 dilewati
sop_retur.txt: 1 chunk baru, 0 dilewati
faq_pengiriman.txt: 1 chunk baru, 0 dilewati
Total baris: 3


### Coba Tebak Dulu
Kalau sel ingest di atas dijalankan **sekali lagi**, berapa jumlah baris di tabel? Jalankan sel di bawah untuk membuktikannya.

In [10]:
sebelum = jumlah_baris()
for doc in raw_documents:
    ingest(doc["text"], doc["source"])
print(f"Sebelum: {sebelum} baris | Sesudah ingest ulang: {jumlah_baris()} baris")

Sebelum: 3 baris | Sesudah ingest ulang: 3 baris


In [11]:
# Tampilkan isi tabel tanpa kolom embedding (384 angka per baris bikin output membengkak)
con.sql("SELECT left(id, 12) AS id, source, content, created_at FROM documents").df()

,id,source,content,created_at
0,583da69c461d,faq_produk.txt,Garansi produk kami berlaku selama 1 tahun sej...,2026-09-25 07:13:03.011
1,56d2936b14a9,sop_retur.txt,Proses retur barang maksimal 7 hari setelah ba...,2026-09-25 07:13:03.403
2,d2ca757ef270,faq_pengiriman.txt,Pengiriman reguler memakan waktu 2-4 hari kerj...,2026-09-25 07:13:03.412


## 7. Fungsi Semantic Search

Query memakai `array_cosine_distance`, sesuai metrik index. Skor yang ditampilkan adalah **similarity = 1 - distance**, jadi makin besar makin relevan.

In [12]:
def search(query, k=3):
    q = embed(query).tolist()
    rows = con.execute(f"""
        SELECT source, content, array_cosine_distance(embedding, ?::FLOAT[{EMBED_DIM}]) AS dist
        FROM documents ORDER BY dist LIMIT ?
    """, [q, k]).fetchall()
    return [{"source": s, "content": c, "similarity": round(1 - d, 4)} for s, c, d in rows]

def tampilkan(query, k=3):
    print(f"Query: {query}")
    for i, r in enumerate(search(query, k), 1):
        print(f"  #{i} sim={r['similarity']:.4f} | {r['source']} | {r['content'][:80]}")
    print()

for q in ["berapa lama garansi produk?", "Pengiriman reguler?", "proses retur?",
          "Berapa lama proses pengiriman reguler", "Berapa harga saham Tesla hari ini?"]:
    tampilkan(q)

Query: berapa lama garansi produk?
  #1 sim=0.7348 | faq_produk.txt | Garansi produk kami berlaku selama 1 tahun sejak tanggal pembelian. Klaim garans
  #2 sim=0.5437 | sop_retur.txt | Proses retur barang maksimal 7 hari setelah barang diterima. Barang harus dalam 
  #3 sim=0.2737 | faq_pengiriman.txt | Pengiriman reguler memakan waktu 2-4 hari kerja. Pengiriman ekspres tersedia unt

Query: Pengiriman reguler?
  #1 sim=0.4716 | faq_pengiriman.txt | Pengiriman reguler memakan waktu 2-4 hari kerja. Pengiriman ekspres tersedia unt
  #2 sim=0.1979 | sop_retur.txt | Proses retur barang maksimal 7 hari setelah barang diterima. Barang harus dalam 
  #3 sim=0.0986 | faq_produk.txt | Garansi produk kami berlaku selama 1 tahun sejak tanggal pembelian. Klaim garans

Query: proses retur?
  #1 sim=0.4716 | sop_retur.txt | Proses retur barang maksimal 7 hari setelah barang diterima. Barang harus dalam 
  #2 sim=0.0691 | faq_pengiriman.txt | Pengiriman reguler memakan waktu 2-4 hari kerja. Pengiriman

Perhatikan skor pertanyaan terakhir (di luar domain). Selisih skor antara query yang relevan dan yang tidak relevan inilah yang nanti dipakai untuk menentukan **similarity threshold** di Tahap 9.

## 8. Ingest PDF

Unggah katalog PDF. Setiap halaman diekstrak lalu di-chunk dengan fungsi yang sama. Coba unggah file yang sama dua kali: jumlah baris tidak akan bertambah.

In [13]:
from google.colab import files
from pypdf import PdfReader
import io

def extract_text_from_pdf(content, filename):
    reader = PdfReader(io.BytesIO(content))
    hasil = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            hasil.append({"source": f"{normalize_source(filename)}_page_{i+1}", "text": text})
    return hasil

print("Silakan unggah file PDF:")
uploaded = files.upload()

for filename, content in uploaded.items():
    sebelum = jumlah_baris()
    for part in extract_text_from_pdf(content, filename):
        ingest(part["text"], part["source"])
    print(f"{filename}: {jumlah_baris() - sebelum} chunk baru | total {jumlah_baris()} baris")

Silakan unggah file PDF:


Saving katalog_produk_emerald.pdf to katalog_produk_emerald.pdf
katalog_produk_emerald.pdf: 13 chunk baru | total 16 baris


In [14]:
tampilkan("ada tinta printer apa aja", 3)
tampilkan("ada produk atk apa aja", 3)

Query: ada tinta printer apa aja
  #1 sim=0.6953 | katalog_produk_emerald.pdf_page_1 | rim. Kertas Foto Glossy A4 230 gsm (EMR-KT-004) Kertas foto mengilap untuk print
  #2 sim=0.5614 | katalog_produk_emerald.pdf_page_2 | Halaman 2 Tinta & Toner Printer Tinta Refill Hitam 100 ml (EMR-TP-001) Tinta ref
  #3 sim=0.5126 | katalog_produk_emerald.pdf_page_2 | dan yellow untuk printer inkjet tipe tangki. Harga: Rp110.000 per set. Cartridge

Query: ada produk atk apa aja
  #1 sim=0.3524 | faq_produk.txt | Garansi produk kami berlaku selama 1 tahun sejak tanggal pembelian. Klaim garans
  #2 sim=0.3255 | katalog_produk_emerald.pdf_page_1 | + Isi Staples (EMR-AT-006) Stapler kecil untuk hingga 20 lembar, termasuk 1 kota
  #3 sim=0.3215 | katalog_produk_emerald.pdf_page_3 | Grosir Pembelian di atas 20 unit untuk produk yang sama mendapat potongan harga 



## 9. RAG dengan Qwen2.5-3B Lokal

Tiga hal yang membuat jawaban model tetap fokus:
- **Chat template.** Qwen2.5-Instruct dilatih dengan format ChatML. `create_chat_completion` membaca template itu dari metadata GGUF, sehingga model tahu kapan gilirannya selesai.
- **Temperature rendah** (0.2) supaya jawaban faktual, bukan kreatif.
- **Similarity threshold.** Kalau tidak ada chunk yang cukup relevan, sistem langsung menjawab "tidak tahu" tanpa memanggil LLM.

`SIM_THRESHOLD` di bawah adalah **nilai awal**. Sesuaikan dengan skor yang muncul di Tahap 7.

In [15]:
import os

MODEL_PATH = "qwen2.5-3b-instruct-q4_k_m.gguf"
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} "https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF/resolve/main/qwen2.5-3b-instruct-q4_k_m.gguf"
print(f"{MODEL_PATH}: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

qwen2.5-3b-instruct-q4_k_m.gguf: 2.10 GB


In [16]:
from llama_cpp import Llama

llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=-1, verbose=False)
print("Qwen2.5-3B siap.")

Qwen2.5-3B siap.


In [17]:
SIM_THRESHOLD = 0.35  # nilai awal, kalibrasi dari skor di Tahap 7
FALLBACK = "Maaf, informasi tersebut tidak ada di knowledge base kami."

SYSTEM_PROMPT = (
    "Kamu adalah asisten customer service. Jawab HANYA berdasarkan konteks yang diberikan, "
    "singkat dan jelas dalam Bahasa Indonesia. Jika jawabannya tidak ada di konteks, "
    f"jawab persis: \"{FALLBACK}\""
)

def rag_answer(pertanyaan, top_k=3, threshold=SIM_THRESHOLD):
    hasil = search(pertanyaan, k=top_k)
    relevan = [h for h in hasil if h["similarity"] >= threshold]

    print(f"Pertanyaan: {pertanyaan}\n")
    print("Hasil retrieval:")
    for h in hasil:
        tanda = "dipakai" if h in relevan else "dibuang"
        print(f"  [{tanda}] sim={h['similarity']:.4f} | {h['source']} | {h['content'][:70]}")

    if not relevan:
        print(f"\nJawaban: {FALLBACK}  (tidak ada chunk di atas threshold, LLM tidak dipanggil)\n")
        return FALLBACK

    konteks = "\n".join(f"- {h['content']}" for h in relevan)
    out = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Konteks:\n{konteks}\n\nPertanyaan: {pertanyaan}"},
        ],
        temperature=0.2,
        max_tokens=256,
    )
    jawaban = out["choices"][0]["message"]["content"].strip()
    u = out["usage"]
    print(f"\nJawaban: {jawaban}")
    print(f"(token prompt={u['prompt_tokens']}, jawaban={u['completion_tokens']}, "
          f"finish_reason={out['choices'][0]['finish_reason']})\n")
    return jawaban

In [18]:
_ = rag_answer("ada produk atk apa aja")
_ = rag_answer("Berapa lama proses pengiriman reguler")
_ = rag_answer("Berapa harga saham Tesla hari ini?")

Pertanyaan: ada produk atk apa aja

Hasil retrieval:
  [dipakai] sim=0.3524 | faq_produk.txt | Garansi produk kami berlaku selama 1 tahun sejak tanggal pembelian. Kl
  [dibuang] sim=0.3255 | katalog_produk_emerald.pdf_page_1 | + Isi Staples (EMR-AT-006) Stapler kecil untuk hingga 20 lembar, terma
  [dibuang] sim=0.3215 | katalog_produk_emerald.pdf_page_3 | Grosir Pembelian di atas 20 unit untuk produk yang sama mendapat poton

Jawaban: Maaf, informasi tersebut tidak ada di knowledge base kami.
(token prompt=129, jawaban=13, finish_reason=stop)

Pertanyaan: Berapa lama proses pengiriman reguler

Hasil retrieval:
  [dipakai] sim=0.6274 | katalog_produk_emerald.pdf_page_3 | Halaman 3 Ketentuan Pembelian Pemesanan Pemesanan dapat dilakukan mela
  [dipakai] sim=0.6052 | faq_pengiriman.txt | Pengiriman reguler memakan waktu 2-4 hari kerja. Pengiriman ekspres te
  [dipakai] sim=0.5826 | katalog_produk_emerald.pdf_page_3 | reguler untuk pembelian di atas Rp500.000. Retur Proses retur barang m


## 10. Simpan Knowledge Base ke Google Drive

File `knowledge.duckdb` dipakai lagi di sesi berikutnya. `CHECKPOINT` memastikan semua perubahan sudah ditulis ke file sebelum disalin.

In [20]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/AI-Engineer/LLM/RAG"  # ubah sesuai folder kamu
os.makedirs(DRIVE_DIR, exist_ok=True)

con.execute("CHECKPOINT")
shutil.copy(DB_PATH, os.path.join(DRIVE_DIR, DB_PATH))
print(f"Tersimpan: {DRIVE_DIR}/{DB_PATH} ({jumlah_baris()} baris)")

Mounted at /content/drive
Tersimpan: /content/drive/MyDrive/AI-Engineer/LLM/RAG/knowledge.duckdb (16 baris)


## Ringkasan
- Knowledge base tersimpan di satu file `knowledge.duckdb` dengan index HNSW (cosine).
- Ingest bersifat idempotent: menjalankan ulang sel atau mengunggah file yang sama tidak menambah baris duplikat.
- Retrieval memakai embedding ternormalisasi dan cosine similarity, sehingga skornya bisa dipakai sebagai threshold.
- RAG memakai chat template, temperature rendah, dan threshold, sehingga pertanyaan di luar domain dijawab "tidak tahu" tanpa memanggil LLM.

---
*Hands-on rubythalib.ai.*